# LatentTokenGuard caption benchmark: cloud run

Runs the object-level hallucination benchmark on a cloud GPU.

**Kaggle:** Settings → Accelerator **GPU T4 x2** · Internet **On**. The full run uses both GPUs.
**Colab:** Runtime → GPU (T4 or A100); run the Drive cell so outputs survive disconnects.

The run is resumable: re-running the run cell skips images already captioned in `OUT`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings
`MODEL`: `llava15_7b_4bit`, `llava15_13b` or `qwen25vl_7b` (all 4-bit) on a 16 GB T4, `llava15_7b` (fp16) on an A100.

In [ ]:
import os
MODEL = "llava15_7b_4bit"      # "llava15_13b", "qwen25vl_7b"; "llava15_7b" on A100
N_IMAGES = 1000                # fixed sample (seed 42), the same on every session
LIMIT = None                   # e.g. 400 to stop after 400 new images in this session
REPO = "https://github.com/sunilsk17/LatentTokenGuard.git"

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/working")
if not ON_KAGGLE:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/ltg_runs/{MODEL}_n{N_IMAGES}_s42"
    WORK = "/content"
else:  # Kaggle: /kaggle/working is saved with the notebook version
    OUT = f"/kaggle/working/{MODEL}_n{N_IMAGES}_s42"
    WORK = "/kaggle/working"
COCO = "/tmp/coco"
print("outputs →", OUT)

## 2. Code and dependencies

In [ ]:
%cd {WORK}
!rm -rf LatentTokenGuard && git clone -q {REPO} LatentTokenGuard
%cd LatentTokenGuard
!git log -1 --oneline
# Do not install requirements.txt here: it pins transformers 4.43 for the old POPE code.
!pip install -q -U bitsandbytes sentencepiece
import transformers, torch; print(transformers.__version__, torch.__version__)

## 3. COCO val2017 (≈1 GB, into /tmp so it is not saved as output)

In [ ]:
import os
if not os.path.exists(f"{COCO}/val2017"):
    !mkdir -p {COCO}
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O {COCO}/val2017.zip
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O {COCO}/ann.zip
    !cd {COCO} && unzip -q val2017.zip && unzip -q ann.zip && rm val2017.zip ann.zip
!ls {COCO}/val2017 | wc -l; ls {COCO}/annotations

## 4. Quick timing check (20 images)
Uses the same output folder, so these 20 count towards the run.

In [ ]:
!python experiments/caption_run.py --model {MODEL} --n_images {N_IMAGES} --limit 20 \
    --images_dir {COCO}/val2017 --annotations_dir {COCO}/annotations --out {OUT}
import json, numpy as np
recs = [json.loads(l) for l in open(f"{OUT}/captions.jsonl")]
gen = np.mean([r["gen_seconds"] + r["feature_seconds"] for r in recs])
print(f"{len(recs)} images, {gen:.1f} s/image → ~{gen * (N_IMAGES - len(recs)) / 3600:.1f} h for the rest")
print("mean tokens:", np.mean([r["n_tokens"] for r in recs]),
      "| mentions/image:", np.mean([len(r["mentions"]) for r in recs]),
      "| hallucinated share:", np.mean([m["label"] for r in recs for m in r["mentions"]] or [0]))
print(recs[0]["caption"][:400])

## 5. Full run (resumable)
With several GPUs (Kaggle T4 x2), one process runs per GPU and each captions every N-th image. The first process loads the model before the others start, so the weights download once. The shard files are merged into `captions.jsonl` at the end.

In [ ]:
import glob, subprocess, time, torch

n_gpu = max(torch.cuda.device_count(), 1)
limit = f"--limit {LIMIT}" if LIMIT else ""
cmd = (f"python experiments/caption_run.py --model {MODEL} --n_images {N_IMAGES} {limit} "
       f"--images_dir {COCO}/val2017 --annotations_dir {COCO}/annotations --out {OUT}")

def started(log):
    return os.path.exists(log) and "Captioning" in open(log, errors="ignore").read()

if n_gpu == 1:
    !{cmd}
else:
    procs = []
    for k in range(n_gpu):
        log = f"{WORK}/shard{k}.log"
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(k)}
        procs.append(subprocess.Popen(f"{cmd} --shard {k}/{n_gpu} > {log} 2>&1", shell=True, env=env))
        while k == 0 and procs[0].poll() is None and not started(log):
            time.sleep(20)
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        done = sum(sum(1 for _ in open(f)) for f in glob.glob(f"{OUT}/captions*.jsonl"))
        print(time.strftime("%H:%M"), f"{done}/{N_IMAGES} images captioned", flush=True)
    for k, p in enumerate(procs):
        print(f"shard {k}: exit code {p.returncode}")
        !tail -c 600 {WORK}/shard{k}.log
    !python experiments/caption_run.py --merge --out {OUT}
!wc -l {OUT}/captions.jsonl

## 6. Evaluate

In [ ]:
!python experiments/caption_eval.py --run {OUT}
from IPython.display import Image, display
display(Image(f"{OUT}/eval/layer_map.png")); display(Image(f"{OUT}/eval/roc_test.png"))

## 7. Save the run
Kaggle: **Save Version → Save & Run All** keeps `/kaggle/working`. To resume in a new session, attach that
version's output as an input dataset and copy the run folder back to `OUT` before step 5.
Colab: the run is already on Drive.

Zip it to download for local evaluation:

In [ ]:
!cd {os.path.dirname(OUT)} && zip -qr {os.path.basename(OUT)}.zip {os.path.basename(OUT)} && ls -lh {os.path.basename(OUT)}.zip

## 8. Hidden states for the linear-probe baseline (after the runs)
Re-reads the stored captions of the published runs (in the repo under `results/caption_runs/`) in one forward pass each and saves the object-token hidden states. Run cells 1–3 (any `MODEL`), then this cell; skip 4–7. About 10–15 min per model on T4 x2. Download the three `hidden_*.zip` files.

In [ ]:
!pkill -f caption_hidden.py   # stop any extraction left running by an earlier attempt
# (run folder in the repo, model key it was made with)
RUNS = [("llava15_7b_4bit_n1000_s42", "llava15_7b_4bit"),
        ("llava15_13b_n1000_s42", "llava15_13b"),
        ("qwen25vl_7b_n1000_s42", "qwen25vl_7b")]
import subprocess, time, torch

n_gpu = max(torch.cuda.device_count(), 1)
for run, model in RUNS:
    out = f"{WORK}/hidden_{run}"
    cmd = (f"python experiments/caption_hidden.py --model {model} --run results/caption_runs/{run} "
           f"--images_dir {COCO}/val2017 --out {out}")
    procs = []
    for k in range(n_gpu):
        log = f"{out}.shard{k}.log"
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(k)}
        procs.append(subprocess.Popen(f"{cmd} --shard {k}/{n_gpu} > {log} 2>&1", shell=True, env=env))
        # the first process downloads the weights before the others start
        while k == 0 and procs[0].poll() is None and not (
                os.path.exists(log) and "Hidden states" in open(log, errors="ignore").read()):
            time.sleep(20)
    for p in procs:
        p.wait()
    n = len([f for f in os.listdir(f"{out}/hidden") if f.endswith(".npz")])
    print(time.strftime("%H:%M"), run, "exit codes", [p.returncode for p in procs], "|", n, "images", flush=True)
    !tail -c 300 {out}.shard0.log
    !cd {WORK} && zip -q -0 -r hidden_{run}.zip hidden_{run} && ls -lh hidden_{run}.zip